## Dados faltantes

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('tabela2296.csv', sep=';',skiprows=3)

print("\n--- Quantidade de Dados Faltantes (Nulos) por Coluna ---")
print(df.isnull().sum())


--- Quantidade de Dados Faltantes (Nulos) por Coluna ---
Brasil e Grande Região     1
setembro 2012             21
outubro 2012              32
novembro 2012             32
dezembro 2012             32
                          ..
outubro 2025              32
novembro 2025             32
dezembro 2025             32
janeiro 2026              32
fevereiro 2026            32
Length: 163, dtype: int64


## Corrigir inconsistências

In [69]:
# Removendo linhas que são praticamente vazias
df = df.dropna(how='all')

# Removendo linhas que não são dados numericos
df = df[df.iloc[:,1:].notnull().sum(axis=1) > 10]

df.isnull().sum()


Brasil e Grande Região    0
setembro 2012             0
outubro 2012              0
novembro 2012             0
dezembro 2012             0
                         ..
outubro 2025              0
novembro 2025             0
dezembro 2025             0
janeiro 2026              0
fevereiro 2026            0
Length: 163, dtype: int64

## Padronizar variáveis

In [70]:
# Limpar nomes das colunas
import unicodedata

def limpar_texto(texto):
    # Converte para minúsculo
    texto = texto.lower()
    
    # Remove espaços no início e fim
    texto = texto.strip()
    
    # Remove acentos (ex: "Região" -> "Regiao")
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    
    # Substitui espaços por underline (padrão de variável)
    texto = texto.replace(' ', '_')
    
    return texto

# Aplica a função em todas as colunas
df.columns = [limpar_texto(col) for col in df.columns]

# Padroniza o nome da coluna para facilitar o uso
df.rename(columns={'brasil_e_grande_regiao': 'regiao'}, inplace=True)

# Coversão de colunas numéricas

# Substitui vírgula por ponto (padrão decimal)
df.iloc[:,1:] = df.iloc[:,1:].replace(',', '.', regex=True)

# Converte todas as colunas (exceto 'regiao') para número
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Transformação para long format
df_long = df.melt(id_vars='regiao', var_name='mes', value_name='custo_m2')

# Converter datas
meses = {
    'janeiro': '01',
    'fevereiro': '02',
    'marco': '03',
    'abril': '04',
    'maio': '05',
    'junho': '06',
    'julho': '07',
    'agosto': '08',
    'setembro': '09',
    'outubro': '10',
    'novembro': '11',
    'dezembro': '12'
}

df_long['mes'] = df_long['mes'].astype(str)

df_long['mes'] = df_long['mes'].apply(
    lambda x: f"{x.split('_')[1]}-{meses.get(x.split('_')[0], '01')}"
)

df_long['mes'] = pd.to_datetime(df_long['mes'], format='%Y-%m')

## Preparar dados para análise

In [ ]:
# Remove linhas onde o custo_m2 está ausente
df_long = df_long.dropna(subset=['custo_m2'])

# Ordena o dataset pela coluna de data
df_long = df_long.sort_values('mes')

# Reorganiza o índice após remoções e ordenações
df_long = df_long.reset_index(drop=True)

# Extrai o ano da coluna de data
df_long['ano'] = df_long['mes'].dt.year

# Extrai o número do mês (1 a 12)
df_long['mes_num'] = df_long['mes'].dt.month

df_long.head()

,regiao,mes,custo_m2,ano,mes_num
0,Brasil,2012-09-01,847.18,2012,9
1,Norte,2012-09-01,478.57,2012,9
2,Brasil,2012-09-01,397.19,2012,9
3,Centro-Oeste,2012-09-01,484.26,2012,9
4,Sul,2012-09-01,431.44,2012,9
